# Examples

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import casadi as cas
from cas_models.continuous_time.models import (
    StateSpaceModelCT,
    SSModelCTLinearFOSISO, 
    SSModelCTLinearO2NoGainSISO,
)
from cas_models.discrete_time.models import StateSpaceModelDTFromCT
from cas_models.discrete_time.simulate import make_n_step_simulation_function_from_model

In [ ]:
# First order single-input, single-output system
sys_model = SSModelCTLinearFOSISO()
print(sys_model.f)
print(sys_model.h)

In [ ]:
# First order SISO system with fixed gain
K = 2.5
sys_model = SSModelCTLinearFOSISO(K=K)
print(sys_model.f)
print(sys_model.h)

In [ ]:
# Second-order system with gain = 1
sys_model_2 = SSModelCTLinearO2NoGainSISO()
print(sys_model_2.f)
print(sys_model_2.h)

In [ ]:
# Combine both systems by connecting in series
sys = sys_model * sys_model_2
print(sys.f)
print(sys.h)

## Example: Water Tank Temperature Control

<img src='images/tank_system_diagram.svg'>

### Case 1. Constant Tank Level

Constraint from Mass Balance

$$F_{out}(t) = F_h(t) + F_c(t)$$

The outflow must equal the total inflow to maintain constant level.

Temperature Dynamics

With $L = L_s$ (constant), the temperature equation becomes:

$$\frac{dT_{out}}{dt} = \frac{F_h(T_h - T_{out}) + F_c(T_c - T_{out})}{A L_s}$$

Expanding:

$$\frac{dT_{out}}{dt} = \frac{F_h T_h + F_c T_c - (F_h + F_c)T_{out}}{A L_s}$$

This is nonlinear due to the product terms $F_h T_{out}$ and $F_c T_{out}$.

### Case 2. With Variable Tank Level

#### Differential Equations

System variables:
- States:
  - $L(t)$ (tank level)
  - $T_{out}(t)$ (outlet/tank temperature)
- Inputs:
  - $F_h(t)$ (hot water inflow)
  - $F_c(t)$ (cold water inflow)
  - $F_{out}(t)$ (outflow)
- Parameters:
  - $A = \frac{\pi D^2}{4}$ (cross-sectional area)
  - $T_h$ (hot inlet temp)
  - $T_c$ (cold inlet temp)

Mass balance:

$$\frac{dL}{dt} = \frac{F_h(t) + F_c(t) - F_{out}(t)}{A}$$

Energy balance:

Assuming a well-mixed tank where $T_{out}(t)$ equals the average tank temperature, the energy balance gives:

$$\rho c_p A \frac{d(L \cdot T_{out})}{dt} = \rho c_p \left(F_h T_h + F_c T_c - F_{out} T_{out}\right)$$

Expanding and substituting the mass balance:

$$A L \frac{dT_{out}}{dt} + T_{out} \frac{dL}{dt} \cdot A = F_h T_h + F_c T_c - F_{out} T_{out}$$

$$A L \frac{dT_{out}}{dt} = F_h(T_h - T_{out}) + F_c(T_c - T_{out})$$

$$\frac{dT_{out}}{dt} = \frac{F_h(T_h - T_{out}) + F_c(T_c - T_{out})}{A L}$$

#### State-Space Model

States: $\mathbf{x} = \begin{bmatrix} L \\ T_{out} \end{bmatrix}$

Inputs: $\mathbf{u} = \begin{bmatrix} F_h \\ F_c \\ F_{out} \end{bmatrix}$

Outputs: $\mathbf{y} = \begin{bmatrix} L \\ T_{out} \end{bmatrix}$

State equations:

$$\frac{d\mathbf{x}}{dt} = \begin{bmatrix} \frac{dL}{dt} \\ \frac{dT_{out}}{dt} \end{bmatrix} = \begin{bmatrix} \frac{F_h
+ F_c - F_{out}}{A} \\ \frac{F_h(T_h - T_{out}) + F_c(T_c - T_{out})}{A L} \end{bmatrix}$$

Output equation:

$$\mathbf{y} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \begin{bmatrix} L \\ T_{out} \end{bmatrix} = \mathbf{x}$$

Note: This is a nonlinear state-space model due to:
1. Division by the state $L$ in the temperature equation
2. Product terms between inputs and states (e.g., $F_h \cdot T_{out}$)

For control design or simulation, you might linearize around an operating point or implement the nonlinear model
directly.

In [ ]:
# Define symbolic variables
t = cas.SX.sym("t")
x = cas.SX.sym("x", 2)  # [L, T_out]
u = cas.SX.sym("u", 3)  # [F_h, F_c, F_out]

# Define parameters
A = cas.SX.sym("A")      # Tank cross-sectional area (m^2)
T_h = cas.SX.sym("T_h")  # Hot water temperature (deg C)
T_c = cas.SX.sym("T_c")  # Cold water temperature (deg C)

# Extract states
L = x[0]        # Tank level (m)
T_out = x[1]    # Outlet/tank temperature (deg C)

# Extract inputs
F_h = u[0]      # Hot water flow rate (m^3/hour)
F_c = u[1]      # Cold water flow rate (m^3/hour)
F_out = u[2]    # Outlet flow rate (m^3/hour)

# State dynamics: dx/dt = f(t, x, u, A, T_h, T_c)
# dL/dt = (F_h + F_c - F_out) / A
dL_dt = (F_h + F_c - F_out) / A

# dT_out/dt = [F_h(T_h - T_out) + F_c(T_c - T_out)] / (A * L)
dT_out_dt = (F_h * (T_h - T_out) + F_c * (T_c - T_out)) / (A * L)

# Combine into rhs vector
rhs = cas.vertcat(dL_dt, dT_out_dt)

# Create f function
f = cas.Function(
    "f",
    [t, x, u, A, T_h, T_c],
    [rhs],
    ["t", "x", "u", "A", "T_h", "T_c"],
    ["rhs"]
)

# Output equation: y = h(t, x, u, A, T_h, T_c)
# y = [L, T_out] = x (both states are measured)
y = x

# Create h function
h = cas.Function(
    "h",
    [t, x, u, A, T_h, T_c],
    [y],
    ["t", "x", "u", "A", "T_h", "T_c"],
    ["y"]
)

# Create the state-space model
tank_model = StateSpaceModelCT(
    f=f,
    h=h,
    n=2,
    nu=3,
    ny=2,
    params={"A": A, "T_h": T_h, "T_c": T_c},
    name="mixing_tank",
    input_names=["F_h", "F_c", "F_out"],
    state_names=["L", "T_out"],
    output_names=["L", "T_out"]
)

print("Continuous-Time System Model:")
print(f"  Name: {tank_model.name}")
print(f"  {tank_model.f}")
print(f"  {tank_model.h}")
print(f"  States (n={tank_model.n}): {tank_model.state_names}")
print(f"  Inputs (nu={tank_model.nu}): {tank_model.input_names}")
print(f"  Outputs (ny={tank_model.ny}): {tank_model.output_names}")
print(f"  Parameters: {list(tank_model.params.keys())}")

In [ ]:
Ts = 0.1  # Sampling time in hours
tank_model_dt = StateSpaceModelDTFromCT(tank_model, dt=Ts)

print("Discrete-Time System Model:")
print(f"  Name: {tank_model_dt.name}")
print(f"  {tank_model_dt.F}")
print(f"  {tank_model_dt.H}")
print(f"  States (n={tank_model.n}): {tank_model_dt.state_names}")
print(f"  Inputs (nu={tank_model_dt.nu}): {tank_model_dt.input_names}")
print(f"  Outputs (ny={tank_model_dt.ny}): {tank_model_dt.output_names}")
print(f"  Parameters: {list(tank_model_dt.params.keys())}")

In [ ]:
nT = 50  # Number of time steps to simulate
simulate_tank = make_n_step_simulation_function_from_model(tank_model_dt, nT)
simulate_tank

In [ ]:
t_eval = np.linspace(0, Ts*nT, nT+1)  # Time evaluation points
x0 = np.array([1.0, 20.0])  # Initial state: [L (m), T_out (deg C)]
U = np.zeros((nT+1, 3))  # Input trajectory: [F_h, F_c, F_out]
U[t_eval >= 1.0, 0] = 0.1  # F_h = 0.1 m^3/hour after t >= 1 hour
U[t_eval >= 2.0, 1] = 0.1  # F_c = 0.2 m^3/hour after t >= 1 hour
U[t_eval >= 1.0, 2] = 0.2  # F_out = 0.3 m^3
D = 0.5
A = np.pi * D**2 / 4  # Tank cross-sectional area (m^2)
T_h = 80.0  # Hot water temperature (deg C)
T_c = 15.0  # Cold water temperature (deg C)
X, Y = simulate_tank(t_eval, U[:-1], x0, A, T_h, T_c)

t_eval = np.array(t_eval).flatten()
X = np.array(X)
Y = np.array(Y)

X.shape, Y.shape

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 4))

ax = axes[0]
ax.plot(t_eval, X[:, 0])
ax.set_ylabel("Tank Level L (m)")
ax.set_title("Tank Level Over Time")
ax.grid()

ax = axes[1]
plt.plot(t_eval, X[:, 1])
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Outlet Temperature T_out (deg C)")
ax.set_title("Outlet Temperature Over Time")
ax.grid()

plt.tight_layout()
plt.show()